In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.certify.randomized import normalize_eigenvalues, axis_lengths
from src.indexing.ner_token_index import load_token_index_artifacts
from src.indexing.base import build_index, query_index
from src.smoothing.pca import fit_local_pca

base = repo_root / "output" / "ner_conll2003_bert"
manifold_last = base / "certify" / "last" / "euclidean" / "annoy" / "index.ann"
iso_last = base / "isotropic_certify" / "last"
token_dir = base / "token_layer_embeddings" / "train" / "last"

SIGMAS_ALL = ["sigma_0_25", "sigma_0_50", "sigma_0_75", "sigma_1_00"]

def sigma_val(s: str) -> float:
    return float(s.replace("sigma_", "").replace("_", "."))

def load_metrics(folder: Path):
    if not folder.exists():
        return None
    for name in ["metrics.json", "running_metrics.json"]:
        p = folder / name
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
            if name == "running_metrics.json":
                return {"_status": "running", "_raw": data}
            return data
    return None

def extract_smooth(m):
    if m is None:
        return None
    if m.get("_status") == "running":
        return m["_raw"].get("running", {})
    return m.get("smoothed", {})

print("Setup complete")
print("repo_root:", repo_root)
print("manifold_last:", manifold_last)
print("iso_last:", iso_last)

# NER New Volume Results Analysis (Job Outputs Only)

This notebook is the **new version** and does not modify your old notebooks.

- Reads completed server outputs
- Uses new volume interpretation (circle vs ellipse, eigen-axis analysis)
- No synthetic experiment generation

## 1) Setup & Paths

Framework used in this notebook:
- Isotropic region: radius = $\sigma$
- Manifold region axes: $a_i = \sigma\sqrt{\tilde\lambda_i}$ with $\tilde\lambda_i = \lambda_i / \lambda_{max}$
- Compare methods from **saved metrics** and **saved eigenvalue arrays**

## 2) Isotropic vs Manifold (from job metrics)

In [ ]:
rows = []
for sig in SIGMAS_ALL:
    sv = sigma_val(sig)

    for method, path in [("Manifold", manifold_last / sig), ("Isotropic", iso_last / sig)]:
        m = load_metrics(path)
        s = extract_smooth(m)
        if s is None:
            continue
        rows.append({
            "sigma": sv,
            "method": method,
            "f1": s.get("f1", np.nan),
            "token_acc": s.get("token_acc", np.nan),
            "certified_acc": s.get("certified_token_acc", np.nan),
            "mean_radius": s.get("mean_certified_radius", np.nan),
            "abstention_rate": s.get("abstention_rate", np.nan),
            "certified_correct": s.get("certified_correct_tokens", np.nan),
            "total_tokens": s.get("total_certified_tokens", np.nan),
        })

if not rows:
    print("No metrics found yet in output folders.")
else:
    df = pd.DataFrame(rows).sort_values(["sigma", "method"])
    print("Summary table:")
    display(df.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for method in ["Manifold", "Isotropic"]:
        sub = df[df["method"] == method].sort_values("sigma")
        axes[0].plot(sub["sigma"], sub["certified_acc"], "o-", linewidth=2, label=method)
        axes[1].plot(sub["sigma"], sub["mean_radius"], "o-", linewidth=2, label=method)
        axes[2].plot(sub["sigma"], sub["abstention_rate"], "o-", linewidth=2, label=method)

    axes[0].set_title("Certified Accuracy vs σ")
    axes[1].set_title("Mean Radius vs σ")
    axes[2].set_title("Abstention vs σ")
    for ax in axes:
        ax.set_xlabel("σ")
        ax.grid(True, alpha=0.3)
        ax.legend()
    axes[0].set_ylabel("Certified Accuracy")
    axes[1].set_ylabel("Mean Radius")
    axes[2].set_ylabel("Abstention Rate")

    plt.suptitle("NER Results from Jobs", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


## 3) Same Accuracy / Same Guarantee Matching

- Same guarantee: compare required $\sigma$
- Same accuracy: compare geometry gain at matched accuracy

In [ ]:
if 'df' not in globals() or df.empty:
    print("Run Section 2 first.")
else:
    mani = df[df["method"] == "Manifold"].sort_values("sigma")
    iso = df[df["method"] == "Isotropic"].sort_values("sigma")

    if mani.empty or iso.empty:
        print("Need both methods present.")
    else:
        matches = []
        for _, mrow in mani.iterrows():
            idx = (iso["certified_acc"] - mrow["certified_acc"]).abs().idxmin()
            irow = iso.loc[idx]
            matches.append({
                "target_acc": float(mrow["certified_acc"]),
                "sigma_mani": float(mrow["sigma"]),
                "sigma_iso_closest": float(irow["sigma"]),
                "abs_gap": float(abs(mrow["certified_acc"] - irow["certified_acc"])),
            })

        df_match = pd.DataFrame(matches).sort_values("target_acc", ascending=False)
        display(df_match.round(4))

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(df_match["target_acc"], df_match["sigma_mani"], "o-", label="σ_mani")
        ax.plot(df_match["target_acc"], df_match["sigma_iso_closest"], "s-", label="σ_iso closest")
        ax.set_xlabel("Target Certified Accuracy")
        ax.set_ylabel("Required σ")
        ax.set_title("Same Guarantee: Required σ at Matched Accuracy")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()


## 4) New Volume Geometry Metrics from `metrics.json`

Reads `volume.geometry` fields from updated pipeline outputs.

In [ ]:
vol_rows = []
for sig in SIGMAS_ALL:
    sv = sigma_val(sig)
    m = load_metrics(manifold_last / sig)
    if not m:
        continue
    vol = m.get("volume", {})
    geo = vol.get("geometry", {}) if isinstance(vol, dict) else {}
    vol_rows.append({
        "sigma": sv,
        "log_v_iso_geo": geo.get("log_v_iso_geo", np.nan),
        "mean_log_v_mani_geo": geo.get("mean_log_v_mani_geo_max_norm", geo.get("mean_log_v_mani_geo", np.nan)),
        "mean_log_geo_ratio": geo.get("mean_log_geo_ratio", np.nan),
        "mean_anisotropy_ratio": geo.get("mean_anisotropy_ratio", np.nan),
        "mean_effective_rank": geo.get("mean_effective_rank", np.nan),
        "k_pca": geo.get("k_pca", vol.get("k_pca", np.nan)),
    })

if not vol_rows:
    print("No updated volume.geometry metrics found yet.")
else:
    df_geo = pd.DataFrame(vol_rows).sort_values("sigma")
    display(df_geo.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(df_geo["sigma"], df_geo["mean_log_geo_ratio"], "o-", linewidth=2)
    axes[0].set_title("Geometry Gain: log(V_mani_geo / V_iso_geo)")
    axes[0].set_xlabel("σ"); axes[0].set_ylabel("mean log geo ratio")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(df_geo["sigma"], df_geo["mean_anisotropy_ratio"], "o-", linewidth=2, color="tab:purple")
    axes[1].set_title("Mean Anisotropy Ratio")
    axes[1].set_xlabel("σ"); axes[1].set_ylabel("a1/ak")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(df_geo["sigma"], df_geo["mean_effective_rank"], "o-", linewidth=2, color="tab:green")
    axes[2].set_title("Mean Effective Rank")
    axes[2].set_xlabel("σ"); axes[2].set_ylabel("effective rank")
    axes[2].grid(True, alpha=0.3)

    plt.suptitle("New Volume Framework Metrics", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


## 5) Full Eigen / Axis Spectrum from `eigenvalues.npz`

Uses full arrays such as `axis_lengths_all`, `anisotropy_ratios`, and geometry gain arrays.

In [ ]:
eigen_path = manifold_last / "sigma_0_50" / "eigenvalues.npz"

if not eigen_path.exists():
    print(f"No eigenvalues file at {eigen_path}")
else:
    data = np.load(eigen_path)
    axis_arr = data.get("axis_lengths_all", data.get("axis_lengths_top10", None))
    anis = data.get("anisotropy_ratios", None)
    geo_ratio = data.get("log_geo_ratio_per_token", data.get("log_geo_ratio_per_sample", None))

    if axis_arr is None:
        print("Axis arrays not found in npz.")
    else:
        axis_arr = np.asarray(axis_arr, dtype=np.float64)
        mean_axes = np.nanmean(axis_arr, axis=0)
        cum = np.cumsum(np.maximum(mean_axes, 0.0))
        if len(cum) > 0 and cum[-1] > 0:
            cum = cum / cum[-1]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(np.arange(1, len(mean_axes) + 1), mean_axes, "o-", linewidth=2, markersize=4)
        axes[0].set_title("Mean Axis Length Spectrum")
        axes[0].set_xlabel("component i"); axes[0].set_ylabel("mean a_i")
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(np.arange(1, len(cum) + 1), cum, "o-", linewidth=2, markersize=4, color="tab:green")
        axes[1].axhline(0.9, linestyle="--", color="orange", alpha=0.8)
        axes[1].axhline(0.95, linestyle="--", color="red", alpha=0.8)
        axes[1].set_title("Cumulative Axis Contribution")
        axes[1].set_xlabel("top components"); axes[1].set_ylabel("cumulative share")
        axes[1].grid(True, alpha=0.3)

        if anis is not None:
            axes[2].hist(np.asarray(anis, dtype=np.float64), bins=30, color="tab:purple", alpha=0.75, edgecolor="black")
            axes[2].set_title("Anisotropy Distribution")
            axes[2].set_xlabel("a1/ak"); axes[2].set_ylabel("count")
        elif geo_ratio is not None:
            axes[2].hist(np.asarray(geo_ratio, dtype=np.float64), bins=30, color="tab:red", alpha=0.75, edgecolor="black")
            axes[2].set_title("Geometry Gain Distribution")
            axes[2].set_xlabel("log_geo_ratio"); axes[2].set_ylabel("count")
        else:
            axes[2].text(0.5, 0.5, "No anisotropy/geo_ratio arrays", ha="center", va="center")

        axes[2].grid(True, alpha=0.3)
        plt.suptitle("Eigen / Axis Analysis from Saved Job Arrays", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

        print("Loaded:", eigen_path)
        print("axis_arr shape:", axis_arr.shape)


## 6) Embedding Neighborhood: Circle vs Ellipse

Direct local-PCA geometry interpretation with new formulas:
- isotropic circle radius = $\sigma$
- manifold ellipse axes = $\sigma\sqrt{\tilde\lambda_i}$

In [ ]:
vectors, token_texts, label_ids = load_token_index_artifacts(token_dir)
index = build_index(vectors=vectors, backend="torch", metric="euclidean")

anchor_idx = 123 if len(vectors) > 124 else 0
anchor_vec = vectors[anchor_idx]
anchor_tok = token_texts[anchor_idx]

sigma = 0.50
knn_k = 200

nids = query_index(index, k=knn_k + 1, vector=anchor_vec)[1:]
neighbor_vecs = vectors[nids]
pca = fit_local_pca(neighbor_vecs)

evals = np.asarray(pca.evals, dtype=np.float64)
evals_norm = normalize_eigenvalues(evals, mode="max")
axes_len = axis_lengths(sigma, evals_norm)

all_pts = np.vstack([anchor_vec.reshape(1, -1), neighbor_vecs])
proj2 = (all_pts - pca.mean) @ pca.evecs[:, :2]
anchor_2d = proj2[0]
nb_2d = proj2[1:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(nb_2d[:, 0], nb_2d[:, 1], c="lightgray", s=18, alpha=0.45, label="kNN")
ax.scatter(anchor_2d[0], anchor_2d[1], c="gold", s=260, marker="*", edgecolors="black", label=f"Anchor: {anchor_tok}")

circle = plt.Circle((anchor_2d[0], anchor_2d[1]), sigma, fill=False, color="blue", linewidth=2.5,
                    label=f"Iso circle (r=σ={sigma})")
ax.add_patch(circle)

width = 2 * axes_len[0]
height = 2 * axes_len[1]
ellipse = Ellipse((anchor_2d[0], anchor_2d[1]), width=width, height=height,
                  fill=False, color="green", linewidth=2.5,
                  label=f"Mani ellipse (a1={axes_len[0]:.3f}, a2={axes_len[1]:.3f})")
ax.add_patch(ellipse)

ax.set_title("Circle vs Ellipse in Local PCA Space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, alpha=0.25)
ax.set_aspect("equal")
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Blue circle = isotropic noise region, same radius in every direction.")
print("- Green ellipse = manifold-shaped region from local eigen-geometry.")
print("- Axes follow a_i = sigma * sqrt(lambda_tilde_i).")
